# Delta Time Travel — Demo Showcase

**Doel:** Laat zien hoe Delta Lake Time Travel werkt aan de hand van de control table
`CONFIG.pipeline_sources`. Deze tabel is een ideaal demo-object omdat:

- Ze bij bootstrap (slice 1) aangemaakt is met twee rijen (versie 0 → 1).
- Slice 2 een `UPDATE … SET load_type = 'incremental'` uitvoert, waardoor een extra versie
  in de Delta-log verschijnt.

**Wat je in dit notebook ziet:**

| Sectie | Patroon |
|--------|---------|
| 1 | Huidige staat van de control table |
| 2 | Versie-query met `VERSION AS OF` |
| 3 | Tijdstip-query met `TIMESTAMP AS OF` |
| 4 | Volledige versiegeschiedenis met `DESCRIBE HISTORY` |

**Wanneer is Time Travel nuttig?**

- **Audit:** Je kunt precies reconstrueren hoe de data eruitzag op een bepaald moment.
- **Rollback:** Bij een fout in een pipeline herstel je de tabel naar de laatste correcte versie.
- **Reproduceerbaarheid:** ML-experimenten, rapporten en dashboards kunnen worden
  vastgepind op een specifieke versie — ook weken later nog.

---

> **Let op:** Dit notebook voert alleen lees-queries uit. Herdraaien is volledig
> non-destructief.

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO_DEV", "Catalog")

catalog = dbutils.widgets.get("catalog")
control_table = f"{catalog}.CONFIG.pipeline_sources"

print(f"Catalog        : {catalog}")
print(f"Control table  : {control_table}")

## Sectie 1 — Huidige staat van de control table

We beginnen met een gewone `SELECT *` om de actuele inhoud van de control table te zien.
Dit is de **meest recente versie** — de HEAD van de Delta-tabel.

In [ ]:
current_df = spark.sql(f"""
    SELECT *
    FROM   {control_table}
    ORDER BY target_table
""")

print(f"Huidige versie van {control_table}:")
print(f"Aantal rijen: {current_df.count()}")
display(current_df)

## Sectie 2 — Vorige versie opvragen met `VERSION AS OF`

Delta Lake bewaart elke schrijfoperatie als een aparte **versie** in de transactielog.
Met `VERSION AS OF <n>` vraag je de exacte snapshot op die na transactie `n` is opgeslagen.

**Praktisch voorbeeld:** Na de bootstrap bevat versie 0 de lege tabel en versie 1 de twee
seed-rijen. Na de `UPDATE` uit slice 2 staat `load_type = 'incremental'` in versie 2.
Met `VERSION AS OF 1` zie je nog de originele `full`-waarden terug.

```sql
SELECT * FROM catalog.CONFIG.pipeline_sources VERSION AS OF 1
```

> **Demo talking point:** Geen back-up nodig — de volledige geschiedenis zit in de
> Delta-transactielog die al in je lakehouse staat.

In [ ]:
# Haal de versiegeschiedenis op om de beschikbare versienummers te tonen.
history_df = spark.sql(f"DESCRIBE HISTORY {control_table}")
versions = [row["version"] for row in history_df.select("version").collect()]

print(f"Beschikbare versies voor {control_table}: {sorted(versions)}")

# Selecteer de vroegste versie die data bevat (versie >= 1 als die bestaat, anders max).
if len(versions) > 1:
    prior_version = sorted(versions)[-2]   # één versie ouder dan de laatste
else:
    prior_version = sorted(versions)[0]    # alleen versie 0 beschikbaar

print(f"\nQuery op versie {prior_version} (VERSION AS OF):")

version_df = spark.sql(f"""
    SELECT *
    FROM   {control_table} VERSION AS OF {prior_version}
    ORDER BY target_table
""")

print(f"Aantal rijen in versie {prior_version}: {version_df.count()}")
display(version_df)

## Sectie 3 — Tijdstip-query met `TIMESTAMP AS OF`

Naast versienummers kun je ook querien op een **exact tijdstip**. Delta zoekt dan de versie
op die geldig was op dat moment — ideaal voor rapportage die je wil reproduceren op basis
van de kalender in plaats van een intern versienummer.

```sql
SELECT * FROM catalog.CONFIG.pipeline_sources TIMESTAMP AS OF '2024-06-01 12:00:00'
```

> **Demo talking point:** Compliance-teams vragen vaak: "Hoe zag de data er op 31 december
> uit voor de jaarafsluiting?" Met `TIMESTAMP AS OF` is dat één query — geen restore nodig.

In [ ]:
from pyspark.sql import functions as F

# Haal het tijdstip van de vroegste versie op uit de history.
# Zo demonstreren we TIMESTAMP AS OF met een tijdstip dat altijd geldig is —
# ongeacht wanneer dit notebook wordt gedraaid.
history_with_ts = spark.sql(f"""
    SELECT version, timestamp
    FROM   (DESCRIBE HISTORY {control_table})
    ORDER BY version ASC
""")

rows = history_with_ts.collect()

# Kies de vroegste versie als tijdstip-ankerpunt.
earliest_row = rows[0]
earliest_version   = earliest_row["version"]
earliest_timestamp = earliest_row["timestamp"]

print(f"Vroegste versie : {earliest_version}")
print(f"Tijdstip        : {earliest_timestamp}")
print(f"\nQuery op tijdstip '{earliest_timestamp}' (TIMESTAMP AS OF):")

# Formatteer het tijdstip als string voor de SQL-query.
ts_str = earliest_timestamp.strftime("%Y-%m-%d %H:%M:%S")

timestamp_df = spark.sql(f"""
    SELECT *
    FROM   {control_table} TIMESTAMP AS OF '{ts_str}'
    ORDER BY target_table
""")

print(f"Aantal rijen op tijdstip '{ts_str}': {timestamp_df.count()}")
display(timestamp_df)

## Sectie 4 — Volledige versiegeschiedenis met `DESCRIBE HISTORY`

`DESCRIBE HISTORY` toont alle wijzigingen die Delta Lake heeft geregistreerd voor deze tabel.
Elke rij is één transactie met:

| Kolom | Betekenis |
|-------|-----------|
| `version` | Oplopend versienummer (0 = aanmaak) |
| `timestamp` | Tijdstip van de schrijfoperatie |
| `operation` | Soort actie: `CREATE TABLE`, `MERGE`, `UPDATE`, `WRITE`, enz. |
| `operationParameters` | Details van de operatie (filter, mode, enz.) |
| `userName` | Wie de wijziging uitvoerde |
| `clusterId` / `notebookId` | Welke compute en welk notebook |

> **Demo talking point:** Dit is de ingebouwde audit-trail van Delta Lake — geen aparte
> logging vereist. Gecombineerd met Unity Catalog Audit Logs heb je een volledig
> auditeerbaar data-platform.

In [ ]:
describe_history_df = spark.sql(f"DESCRIBE HISTORY {control_table}")

# Toon de meest relevante kolommen voor de demo.
history_display_df = describe_history_df.select(
    "version",
    "timestamp",
    "operation",
    "operationParameters",
    "userName",
    "clusterId",
    "notebookId",
).orderBy("version")

print(f"Versiegeschiedenis van {control_table}:")
print(f"Totaal aantal versies: {history_display_df.count()}")
display(history_display_df)

## Samenvatting — Delta Time Travel patronen

| Patroon | SQL-syntax | Wanneer gebruiken |
|---------|------------|-------------------|
| `VERSION AS OF` | `SELECT * FROM tabel VERSION AS OF 1` | Je weet het exacte versienummer (bijv. na een specifieke pipeline-run) |
| `TIMESTAMP AS OF` | `SELECT * FROM tabel TIMESTAMP AS OF '2024-01-01'` | Je weet het tijdstip maar niet het versienummer |
| `DESCRIBE HISTORY` | `DESCRIBE HISTORY tabel` | Overzicht van alle wijzigingen en wie ze uitvoerde |

**Alle queries hierboven zijn read-only** — de control table is niet gewijzigd.
Herdraaien van dit notebook geeft altijd hetzelfde resultaat.

---

### Volgende stap

Bekijk `demo_showcase/audit_logs.ipynb` voor de Unity Catalog Audit Logs demo:
wie heeft wat wanneer gewijzigd — op workspace-niveau.